# Curation of Weather Data

## 1. Objective

For both Crop and Weather data, produce clean Pandas dataframe ready for use by analytics methods.

## 1. Plan

- Configure Environment.
- Retrieve weather data from URL.
  - Raw data, with unnecessary metadata removed.
- Download and save as csv for offline use.
- Parse csv with Pandas dataframe and prepare analytics.

#### Test Kernel is working

In [ ]:
print ("Hello, World!")

#### Import relevant libraries

Dependencies needed for the functions in this project.

In [ ]:
import os
import requests
import pandas as pd
import sqlite3

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

## 2. Ingestion of Weather Data

- Get data via url.
- Paramatise the url to get based on wether type and region.
- Parse into DataFrame.

### Get data from url

For the purposes of retrieving weather data by year, to select the data to download, two parameters are passed: the **Weather Type**, and the **Area**. This information was manually extracted from the html of the website: <https://www.metoffice.gov.uk/research/climate/maps-and-data/uk-and-regional-series>

The for requests follows the format <https://www.metoffice.gov.uk/pub/data/weather/uk/climate/datasets/{weather}/date/{area}.txt>. Note the **weather** and **area** parameters in curly braces. These are used to determine which specific dataset is requested.

#### Weather Types

These are the parameters used.

|Parameter|Label|
|---------|-----|
|Max temp|Tmax|
|Min temp|Tmin|
|Mean temp|Tmean|
|Sunshine|Sunshine|
|Rainfall|Rainfall|
|Rain days ≥1.0mm|Raindays1mm|
|Days of air frost|AirFrost|

This "weather_types" list  can be used to iterate over to retrieve all weather types.

In [ ]:
weather_types = ["Tmax", "Tmin", "Tmean", "Sunshine", "Rainfall", "Raindays1mm", "AirFrost"]

#### Areas

These Regions are how the data for each weather type is aggregated and segmented.

|ID|Region|Label|
|--|------|-----|
|1|UK|UK|
|2|England|England|
|3|Wales|Wales|
|4|Scotland|Scotland|
|5|Northern Ireland|Northern_Ireland|
|6|England & Wales|England_and_Wales|
|7|England N|England_N|
|8|Englan S|Englan_S|
|9|Scotland N|Scotland_N|
|10|Scotland E|Scotland_E|
|11|Scotland W|Scotland_W|
|12|England E & NE|England_E_and_NE|
|13|England NW/Wales N|England_NW_and_N_Wales|
|14|Midlands|Midlands|
|15|East Anglia|East_Anglia|
|16|England SW/Wales S|England_SW_and_S_Wales|
|17|England SE/Central S|England_SE_and_Central_S|

This "areas" list can be used to iterate to request each of the areas in turn for each of the weather types previously noted.

In [ ]:
areas = ["UK",
		"England",
		"Wales",
		"Scotland",
		"Northern_Ireland",
		"England_and_Wales",
		"England_N",
		"England_S",
		"Scotland_N",
		"Scotland_E",
		"Scotland_W",
		"England_E_and_NE",
		"England_NW_and_N_Wales",
		"Midlands",
		"East_Anglia",
		"England_SW_and_S_Wales",
		"England_SE_and_Central_S"]

print (areas)

### Parse into a csv

Rather than downloading each of file for each region for each parameter, it would be better to directly retrieve the data from the API or URL.

The URL clearly indicates in plain text the Region and Parameter labels. I can extract these and inject them into the URL string used to request the data.

In [ ]:
# For each Weather Type, for each Area, request data from the met office website.
for weather in weather_types:
	for area in areas:

		# Ensure that the directory exists before writing files
		os.makedirs(f'data/weather/{weather}', exist_ok=True)

		# Get data with parametrised URL.
		url= f"https://www.metoffice.gov.uk/pub/data/weather/uk/climate/datasets/{weather}/date/{area}.txt"
		response = requests.get(url)
		print (response.text)

		# Read with flexible whitespace handling.
		try:
			df = pd.read_csv(url, 
							 sep='\s+',					# Handle variable spacing
							 skipinitialspace=True,		# Skip leading spaces
							 skip_blank_lines=True,		# Skip empty lines
							 skiprows=5)				# Skip the first 5 lines of metadata

			# Write to CSV with a filename that includes the weather type and area.
			df.to_csv(f"data/weather/{weather}/{areas}_{weather}.csv", index=False)

		# Catch any exceptions.
		except Exception as e:
			print(f"Error: {str(e)}")

### Save to SQLite

#### Prerequisites for SQLite database

In [ ]:
os.makedirs('data', exist_ok=True)
project_db  = 'data/project_db.db'

#### Populate SQLite with data directly from source URL.

Rather than downloading each of file for each region for each parameter, it would be better to directly retrieve the data from the API or URL.

The URL clearly indicates in plain text the Region and Parameter labels. I can extract these and inject them into the URL string used to request the data.

In [ ]:
conn = sqlite3.connect(project_db)

for weather in weather_types:
	for i, area in enumerate(areas):

		# Get data with parametrised URL.
		url = f"https://www.metoffice.gov.uk/pub/data/weather/uk/climate/datasets/{weather}/date/{area}.txt"

		# Read with flexible whitespace handling.
		try:
			df = pd.read_csv(url,
							sep='\s+',					# Handle variable spacing
							skipinitialspace=True,		# Skip leading spaces
							skip_blank_lines=True,		# Skip empty lines
							skiprows=5,					# Skip the first 5 lines of metadata
							na_values=['---'],			# Specifies which value indicates null/missing data.
							keep_default_na=True,		# Retain default NA values.
							dtype={'year':int}
			)

			# Add an 'area' column to the DataFrame so this can be queried independently.
			df['area'] = area

			# Replace the table on the first area (clears old data),
			# then append for the rest — avoids duplicates on re-runs.
			write_mode = 'replace' if i == 0 else 'append'

			df.to_sql(
				name=weather,
				con=conn,
				if_exists=write_mode,
				index=False
			)

			# Write to console to monitor progress.
			print(f"Table {weather} / {area} updated.")

		# Write to console to show if any errors occur, but continue processing the rest of the data.
		except Exception as e:
			print(f"Error with updating table ({weather}/{area}): {str(e)}.")

conn.close()

print("All tables updated successfully.")

#### Sample SQLite Query

In [ ]:
conn = sqlite3.connect(project_db)

df_rainfall_all = pd.read_sql('SELECT * FROM Rainfall ', conn)
conn.close()

df_rainfall_all

In [ ]:
df_rainfall_all.dtypes

In [ ]:
conn = sqlite3.connect(project_db)

df_rainfall_england = pd.read_sql('SELECT * FROM Rainfall WHERE area = "England"', conn)
conn.close()

df_rainfall_england

In [ ]:
conn = sqlite3.connect(project_db)

df = pd.read_sql('SELECT \
					year, spr, sum, aut, win, ann \
				FROM \
					Rainfall \
				WHERE \
					area = "England" \
					 and year >= 1999 and year <= 2025'
				, conn)
conn.close()

df

In [ ]:
conn = sqlite3.connect(project_db)

query = """
    SELECT 
        r.year,
        r.ann AS Rainfall,
        t.ann AS MaxTemp,
        s.ann AS Sunshine
    FROM Rainfall r
    JOIN Tmax t ON r.year = t.year
    JOIN Sunshine s ON r.year = s.year
    WHERE r.area = 'England'
      AND t.area = 'England'
      AND s.area = 'England'
	  AND r.year >= 1999 AND r.year <= 2025
"""

df_weather_england_annual = pd.read_sql(query, conn)
conn.close()
df_weather_england_annual

## 5. Cleansed Dataframe

|Region|Year|Sun|Rain|Temp|Crop|
|------|----|---|----|----|----|
|UK|1999|23|25|525|4535|
|England|2000|23|25|525|4535|
|Scotland|2000|23|25|525|4535|
|Wales|2000|23|25|525|4535|

### Sunshine for each month, season and annually per year from 1999 to 2025 inclusive.